# 01 — Découverte du dataset

## Objectif

Ce notebook présente une première exploration du dataset  
« Diabetes 130-US Hospitals for Years 1999–2008 ».

Les objectifs sont :

- charger le dataset ;
- vérifier sa structure ;
- identifier les colonnes ;
- analyser les types de données ;
- étudier les principales statistiques ;
- comprendre la variable cible `readmitted` ;
- préparer les futures étapes de qualité et de transformation.

In [ ]:
from pathlib import Path

import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "source" / "diabetic_data.csv"
MAPPING_PATH = PROJECT_ROOT / "data" / "source" / "IDS_mapping.csv"

print(f"Dataset principal : {DATA_PATH}")
print(f"Fichier de mapping : {MAPPING_PATH}")

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "source" / "diabetic_data.csv"
MAPPING_PATH = PROJECT_ROOT / "data" / "source" / "IDS_mapping.csv"

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Le fichier diabetic_data.csv est introuvable : {DATA_PATH}"
    )

print("Le fichier principal existe.")

In [ ]:
if MAPPING_PATH.exists():
    print("Le fichier IDS_mapping.csv existe.")
else:
    print("Le fichier IDS_mapping.csv n'a pas encore été ajouté.")

In [ ]:
df = pl.read_csv(
    DATA_PATH,
    null_values=["?", "Unknown/Invalid", "NULL", "None", ""],
    infer_schema_length=10000,
)

df.head()

In [ ]:
null_values=["?", "Unknown/Invalid", "NULL", "None", ""]

In [ ]:
df_raw = pl.read_csv(
    DATA_PATH,
    infer_schema_length=10000,
)

In [ ]:
row_count = df.height
column_count = df.width

print(f"Nombre de lignes : {row_count:,}")
print(f"Nombre de colonnes : {column_count}")

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
df.tail(5)

In [ ]:
for index, column in enumerate(df.columns, start=1):
    print(f"{index:02d}. {column}")

In [ ]:
schema_df = pl.DataFrame(
    {
        "column": df.columns,
        "dtype": [str(dtype) for dtype in df.dtypes],
    }
)

schema_df

In [ ]:
df.schema

In [ ]:
numeric_columns = [
    column
    for column, dtype in df.schema.items()
    if dtype.is_numeric()
]

categorical_columns = [
    column
    for column, dtype in df.schema.items()
    if dtype == pl.String
]

print("Colonnes numériques :")
print(numeric_columns)

print("\nColonnes textuelles :")
print(categorical_columns)

In [ ]:
identifier_columns = [
    "encounter_id",
    "patient_nbr",
]

coded_categorical_columns = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]

target_column = "readmitted"

In [ ]:
df.describe()

In [ ]:
df.select(
    [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]
).describe()

In [ ]:
df.select(
    pl.col("readmitted").value_counts(sort=True)
)

In [ ]:
df.select(
    pl.col("age").value_counts(sort=True)
)

In [ ]:
df.select(
    pl.col("gender").value_counts(sort=True)
)

In [ ]:
unique_counts = pl.DataFrame(
    {
        "column": df.columns,
        "unique_count": [
            df.select(pl.col(column).n_unique()).item()
            for column in df.columns
        ],
    }
).sort("unique_count")

unique_counts

In [ ]:
constant_columns = unique_counts.filter(
    pl.col("unique_count") <= 1
)

constant_columns

In [ ]:
target_distribution = (
    df.group_by("readmitted")
    .agg(pl.len().alias("count"))
    .with_columns(
        (
            pl.col("count") / pl.col("count").sum() * 100
        ).round(2).alias("percentage")
    )
    .sort("count", descending=True)
)

target_distribution

In [ ]:
df_target_preview = df.with_columns(
    pl.when(pl.col("readmitted") == "<30")
    .then(1)
    .otherwise(0)
    .alias("readmitted_30_days")
)

df_target_preview.select(
    pl.col("readmitted_30_days").value_counts(sort=True)
)

In [ ]:
df_target_preview.group_by("readmitted_30_days").agg(
    pl.len().alias("count")
).with_columns(
    (
        pl.col("count") / pl.col("count").sum() * 100
    ).round(2).alias("percentage")
)

In [ ]:
target_pd = target_distribution.to_pandas()

plt.figure(figsize=(8, 5))
plt.bar(target_pd["readmitted"], target_pd["count"])
plt.title("Distribution de la variable readmitted")
plt.xlabel("Statut de réhospitalisation")
plt.ylabel("Nombre d'hospitalisations")
plt.show()

In [ ]:
unique_encounters = df.select(
    pl.col("encounter_id").n_unique()
).item()

print(unique_encounters)

In [ ]:
print(f"Lignes : {df.height}")
print(f"Encounter IDs uniques : {unique_encounters}")

In [ ]:
print(f"Lignes : {df.height}")
print(f"Encounter IDs uniques : {unique_encounters}")

In [ ]:
print(f"Lignes : {df.height}")
print(f"Encounter IDs uniques : {unique_encounters}")

In [ ]:
unique_patients = df.select(
    pl.col("patient_nbr").n_unique()
).item()

print(f"Nombre de patients uniques : {unique_patients:,}")

In [ ]:
average_encounters_per_patient = df.height / unique_patients

print(
    f"Nombre moyen de séjours par patient : "
    f"{average_encounters_per_patient:.2f}"
)

In [ ]:
patient_encounters = (
    df.group_by("patient_nbr")
    .agg(pl.len().alias("encounter_count"))
    .sort("encounter_count", descending=True)
)

patient_encounters.head(20)

In [ ]:
patients_multiple_encounters = patient_encounters.filter(
    pl.col("encounter_count") > 1
).height

print(
    f"Patients avec plusieurs séjours : "
    f"{patients_multiple_encounters:,}"
)

In [ ]:
main_numeric_columns = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

In [ ]:
df.select(main_numeric_columns).describe()

In [ ]:
time_in_hospital_pd = df.select(
    "time_in_hospital"
).to_pandas()

plt.figure(figsize=(8, 5))
plt.hist(
    time_in_hospital_pd["time_in_hospital"],
    bins=14,
    edgecolor="black",
)
plt.title("Distribution de la durée d'hospitalisation")
plt.xlabel("Nombre de jours")
plt.ylabel("Nombre de séjours")
plt.show()

In [ ]:
def categorical_summary(
    dataframe: pl.DataFrame,
    column: str,
    limit: int = 20,
) -> pl.DataFrame:
    return (
        dataframe.group_by(column)
        .agg(pl.len().alias("count"))
        .with_columns(
            (
                pl.col("count") / pl.col("count").sum() * 100
            ).round(2).alias("percentage")
        )
        .sort("count", descending=True)
        .head(limit)
    )

In [ ]:
categorical_summary(df, "race")

In [ ]:
categorical_summary(df, "gender")

In [ ]:
categorical_summary(df, "age")

In [ ]:
categorical_summary(df, "medical_specialty")

In [ ]:
categorical_summary(df, "A1Cresult")

In [ ]:
categorical_summary(df, "max_glu_serum")

In [ ]:
categorical_summary(df, "insulin")

In [ ]:
categorical_summary(df, "change")

In [ ]:
categorical_summary(df, "diabetesMed")

In [ ]:
df.select(
    ["diag_1", "diag_2", "diag_3"]
).head(20)

In [ ]:
df.select(
    [
        pl.col("diag_1").n_unique().alias("diag_1_unique"),
        pl.col("diag_2").n_unique().alias("diag_2_unique"),
        pl.col("diag_3").n_unique().alias("diag_3_unique"),
    ]
)

In [ ]:
categorical_summary(df, "diag_1", limit=20)

In [ ]:
medication_columns = [
    "metformin",
    "repaglinide",
    "nateglinide",
    "chlorpropamide",
    "glimepiride",
    "acetohexamide",
    "glipizide",
    "glyburide",
    "tolbutamide",
    "pioglitazone",
    "rosiglitazone",
    "acarbose",
    "miglitol",
    "troglitazone",
    "tolazamide",
    "examide",
    "citoglipton",
    "insulin",
    "glyburide-metformin",
    "glipizide-metformin",
    "glimepiride-pioglitazone",
    "metformin-rosiglitazone",
    "metformin-pioglitazone",
]

In [ ]:
existing_medication_columns = [
    column
    for column in medication_columns
    if column in df.columns
]

missing_medication_columns = [
    column
    for column in medication_columns
    if column not in df.columns
]

print("Colonnes disponibles :", existing_medication_columns)
print("Colonnes absentes :", missing_medication_columns)

In [ ]:
categorical_summary(df, "metformin")

In [ ]:
categorical_summary(df, "insulin")